In [1]:
import os
import openai
from dotenv import load_dotenv
from langchain.vectorstores import Chroma
from langchain.embeddings import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain.schema.messages import SystemMessage, HumanMessage, AIMessage
from langchain_core.prompts import ChatPromptTemplate, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain.schema import Document


In [2]:
load_dotenv()
os.environ["OPEN_API_KEY"] = os.getenv("OPENAI_API_KEY")
openai.api_key = os.getenv("OPENAI_API_KEY")

In [22]:
def classify_query(query: str, llm: ChatOpenAI) -> str:
    """
    질의를 분석해 필요한 정보 DB를 결정합니다.
    가능한 반환 값: "edu", "news", "report", "all"
    """
    prompt = ChatPromptTemplate.from_messages([
        ('system',
        """
        다음 질의를 읽고, 관련된 정보 유형을 한 단어로 출력해줘.
        가능한 답변은 "edu", "news", "report", "all", "nothing" 중 하나야.
        클래스에 대한 설명은 다음과 같아.
        
        - edu: 경제 용어, 상황 등에 대한 전반적인 배경지식을 설명
        - news: 현재 산업 전반에 대한 관련 최신 정보 포함
        - report: 기업에 관련된 더 깊고 전문적인 지식 포함, 기업과 기업간의 관계 및 현재 재무재표 관련 정보 포함
        - all/nothing: 이 중에 특정할 수 없는 쿼리로 구성

        "nothing"과 "all"로 분류할 확률은 최소한으로 하고,
        최대한 "edu", "news", "report" 중에 분류해줘."""),
        ('user', """질의: {query}
        답변:
        """)
    ])

    chain = prompt | llm
    classification = chain.invoke({"query": query}).content.strip().lower()
    return classification

In [26]:

def load_vector_store(persist_directory: str) -> Chroma:
    """
    지정된 persist_directory에 있는 Chroma DB를 불러옵니다.
    """
    return Chroma(
        persist_directory=persist_directory,
        embedding_function=OpenAIEmbeddings()
    )

def run_query(query: str) -> str:
    """
    주어진 질의에 대해 세 개의 DB(교육, 뉴스, 리포트)에서 유사 문서를 검색하고,
    이를 바탕으로 청소년용 주식 교육 및 추천 답변을 생성합니다.
    """
    # 각 DB 로드
    edu_db = load_vector_store("chroma_edu_db")
    news_db = load_vector_store("chroma_news_db")
    report_db = load_vector_store("chroma_report_db")
    
    llm = ChatOpenAI(model_name="gpt-4o", temperature=0.2, max_tokens=1024)
    classification = classify_query(query, llm)
    print(f"질의 분류 결과: {classification}")

    combined_info = ""
    # 질의와 유사한 문서를 각 DB에서 검색 (각각 상위 3개)
    if classification in ["edu", "all"]:
        edu_docs = edu_db.similarity_search(query, k=3)
        if edu_docs:
            combined_info += "【유튜브 경제 교육 자료 (청소년용)】\n" + "\n".join([doc.page_content for doc in edu_docs]) + "\n\n"
    if classification in ["news", "all"]:
        news_docs = news_db.similarity_search(query, k=3)
        if news_docs:
            combined_info += "【경제 관련 뉴스】\n" + "\n".join([doc.page_content for doc in news_docs]) + "\n\n"
    if classification in ["report", "all"]:
        report_docs = report_db.similarity_search(query, k=3)
        if report_docs:
            combined_info += "【증권 보고서】\n" + "\n".join([doc.page_content for doc in report_docs]) + "\n\n"

    if classification in ["nothing"]:
        prompt_messages = [
            SystemMessage(content=
            f"""
            너는 주식 교육 및 추천 종목에 관한 정보를 제공하는 경제 전문가이자 청소년 맞춤 경제 교육 챗봇이야.
            만약 사용자가 입력한 질의가 경제 관련 단어라면 사용자가 입력한 질의에 대해 청소년도 쉽게 이해할 수 있도록 설명하고, 말투는 존댓말을 유지해줘.
            사용자가 입력한 질의가 전혀 다른 문맥이라면 '경제와 관련된 질문을 입력해주세요!' 라고 답을 해줘.
            """),
        HumanMessage(content=
            f"""
            질의: {query}

            답변:
            """)
        ]

        chatbot_prompt = ChatPromptTemplate.from_messages(prompt_messages)
        chatbot_chain = chatbot_prompt | llm
        answer = chatbot_chain.invoke({"query": query}).content

        return answer

    prompt_messages = [
        SystemMessage(content=
            f"""
            너는 주식 교육 및 추천 종목에 관한 정보를 제공하는 경제 전문가이자 청소년 맞춤 경제 교육 챗봇이야.
            아래는 너가 참고할 수 있도록 각각의 DB에서 검색된 정보들이야:

            {combined_info}

            이 정보를 기반으로 사용자가 입력한 질의에 대해 청소년도 쉽게 이해할 수 있도록 설명하고,
            특히 뉴스를 기반으로 정보를 추출한다면 최근 현황에 대해 설명해줘.
            필요하다면 투자 추천 종목도 함께 알려줘.
            말투는 존댓말을 유지해줘.
            """),
        HumanMessage(content=
            f"""
            질의: {query}

            답변:
            """)
    ]

    # ChatOpenAI를 사용해 답변 생성 (모델 및 온도 설정은 필요에 따라 조정)
    chatbot_prompt = ChatPromptTemplate.from_messages(prompt_messages)

    chatbot_chain = chatbot_prompt | llm

    answer = chatbot_chain.invoke({"combined_info": combined_info, "query": query}).content

    # response = llm([{"role": "user", "content": prompt}])
    # answer = response.choices[0].message.content.strip()
    return answer

if __name__ == "__main__":
    user_query = input("질의를 입력하세요: ")
    print("\n질의:\n", user_query)
    result = run_query(user_query)
    print("\n답변:\n", result)



질의:
 현재 환율
질의 분류 결과: news

답변:
 현재 원-달러 환율은 1,470원을 넘어서며 약 50일 만에 다시 높은 수준에 도달했습니다. 환율이 높아지면 해외에서 물건을 사거나 여행할 때 더 많은 돈이 필요하게 됩니다. 이는 경제에 영향을 미칠 수 있어 많은 사람들이 주목하고 있습니다. 환율이 이렇게 높아진 이유는 여러 가지가 있을 수 있지만, 주로 글로벌 경제 상황이나 국내외 경제 정책 등이 영향을 미칩니다. 

환율이 높을 때는 수출 기업에게는 유리할 수 있지만, 수입 기업이나 해외 여행객에게는 불리할 수 있습니다. 따라서 환율 변동에 따라 경제 전반에 다양한 영향을 미칠 수 있습니다. 주식 투자에 관심이 있다면, 환율 상승으로 이익을 볼 수 있는 수출 관련 기업에 주목해보는 것도 좋습니다. 예를 들어, 자동차나 전자제품을 해외로 많이 수출하는 기업들이 이에 해당할 수 있습니다.
